# Preprocesamiento e ingeniería de características — UNSW-NB15

Este notebook implementa la etapa de preprocesamiento del proyecto. Traduce los hallazgos del
análisis exploratorio (`eda.ipynb`) a un espacio de características apto para el modelado
multiclase, y produce las particiones de entrenamiento, validación y prueba.

**Resultado esperado:** 2 540 047 filas × 45 variables predictoras, repartidas en tres
conjuntos disjuntos guardados en `output/preprocessed/`.

## Decisiones y su justificación

Cada decisión de este notebook responde a un hallazgo concreto del EDA. El detalle argumentado
está en `.agents/PROPUESTA_PREPROCESAMIENTO.md`.

| Decisión | Hallazgo que la motiva |
|---|---|
| **Se conservan los duplicados** | El 88.22 % de `Generic` y el 65.36 % de `DoS` son filas duplicadas exactas, frente al 11.67 % de `Benign`. Eliminarlas llevaría `Benign` del 87.35 % al 95.16 % y eliminaría el 88 % de la segunda familia de ataques más numerosa. |
| **Los blancos estructurales se imputan a 0** | El blanco sigue al archivo de origen (0 %, 43.86 %, 98.49 %, 98.49 %) y no al protocolo del flujo. Codificarlo como categoría haría que el modelo dedujese la procedencia del registro. |
| **Los puertos se derivan a rangos** | `Benign` usa 64 581 puertos distintos y `Analysis` solo 3. El valor crudo permite memorizar «puerto 6881 → Generic». |
| **`Stime` y `Ltime` se eliminan** | Describen el instante de la captura, no el comportamiento del flujo. Se usan solo para ordenar y particionar. |
| **La partición es cronológica** | Con partición aleatoria, el contexto de 100 conexiones que sostiene cada variable `ct_*` quedaría repartido entre train y test, y el modelo memorizaría ráfagas en lugar de detectar intrusiones. |
| **`log1p` se decide sobre train** | Calcular la asimetría sobre las 2.54 M filas completas incorporaría estadísticas de test en la transformación de train. |

## Orden de ejecución

El orden de las celdas es **obligatorio**. En particular, la imputación por moda y la
selección de `log1p` dependen de conocer el conjunto de entrenamiento, por lo que la
partición se calcula antes que ambas.

## 0. Dependencias

In [ ]:
%pip install pandas numpy pyarrow kagglehub

## 1. Configuración y carga de datos

Se lee el dataset completo desde `dataset/`, descargándolo desde Kaggle si no está presente.
Los `dtypes` de ancho reducido (`Int8`, `Int16`, `Float32`, `boolean`) son esenciales: sin
ellos el conjunto no cabe en memoria. El diccionario `NA_VALUES` marca como `NaN` los tokens
corruptos de la fuente (hexadecimales ilegibles en los puertos, y los valores `2` y `4` de
`is_ftp_login`, que deberían ser binarios).

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path('dataset') if Path('dataset').is_dir() else Path('../dataset')
OUT_DIR = Path('output') if Path('output').is_dir() else Path('../output')
PROC_DIR = OUT_DIR / 'preprocessed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = 'harshwardhanbhangale/unsw-complete-dataset'
REQ = [f'UNSW-NB15_{i}.csv' for i in range(1, 5)]
if all((DATA_DIR / f).exists() for f in REQ):
    print('Dataset ya presente en', DATA_DIR)
else:
    print('Descargando dataset desde Kaggle...')
    import kagglehub
    tmp = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for f in REQ:
        origen = tmp / f
        if origen.exists():
            shutil.copy2(origen, DATA_DIR / f)
            print('  ->', f)

In [ ]:
COLUMNS = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state',
    'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss',
    'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin',
    'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
    'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt',
    'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl',
    'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src',
    'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label',
]

DTYPES = {
    'srcip': 'string', 'dstip': 'string', 'proto': 'string',
    'state': 'string', 'service': 'string', 'attack_cat': 'string',
    'sport': 'Int32', 'dsport': 'Int32', 'sbytes': 'Int32', 'dbytes': 'Int32',
    'sttl': 'Int16', 'dttl': 'Int16', 'sloss': 'Int16', 'dloss': 'Int16',
    'Spkts': 'Int16', 'Dpkts': 'Int16', 'swin': 'Int16', 'dwin': 'Int16',
    'stcpb': 'Int64', 'dtcpb': 'Int64', 'smeansz': 'Int16', 'dmeansz': 'Int16',
    'trans_depth': 'Int16', 'res_bdy_len': 'Int32', 'Stime': 'Int32', 'Ltime': 'Int32',
    'ct_state_ttl': 'Int8', 'ct_flw_http_mthd': 'Int8', 'ct_ftp_cmd': 'Int8',
    'ct_srv_src': 'Int8', 'ct_srv_dst': 'Int8', 'ct_dst_ltm': 'Int8',
    'ct_src_ ltm': 'Int8', 'ct_src_dport_ltm': 'Int8', 'ct_dst_sport_ltm': 'Int8',
    'ct_dst_src_ltm': 'Int8',
    'dur': 'Float32', 'Sload': 'Float64', 'Dload': 'Float64',
    'Sjit': 'Float32', 'Djit': 'Float32', 'Sintpkt': 'Float32', 'Dintpkt': 'Float32',
    'tcprtt': 'Float32', 'synack': 'Float32', 'ackdat': 'Float32',
    'is_sm_ips_ports': 'boolean', 'is_ftp_login': 'boolean', 'Label': 'boolean',
}

# Tokens corruptos de la fuente original, marcados intencionalmente como NaN.
NA_VALUES = {
    'sport': ['0x000b', '0x000c', '-'],
    'dsport': ['0xc0a8', '0xcc09', '0x20205321', '-'],
    'ct_ftp_cmd': [' '],
    'is_ftp_login': ['2', '4'],
    'state': ['no'],
}

df = pd.concat(
    [pd.read_csv(DATA_DIR / f'UNSW-NB15_{i}.csv', header=None, names=COLUMNS,
                 dtype=DTYPES, na_values=NA_VALUES, low_memory=False)
     for i in range(1, 5)],
    ignore_index=True,
)
N = len(df)
print(f'Shape: {N:,} filas x {len(df.columns)} columnas')

In [ ]:
# Definición del objetivo multiclase. Los registros sin attack_cat son tráfico normal,
# que se agrupa en la décima clase 'Benign'. La etiqueta original usa 'Backdoor' en
# singular; se unifica con 'Backdoors' para que la etiqueta sea consistente.
df['attack_cat'] = (
    df['attack_cat'].fillna('Benign').str.strip().replace({'Backdoor': 'Backdoors'})
)

# Verificación 1:1 entre la etiqueta multiclase y la binaria. Si no se cumple, la
# clase Benign y el objetivo no serían consistentes y la tarea estaría mal definida.
inconsistente = int(((df['attack_cat'] != 'Benign') & (df['Label'] == 0)).sum())
if inconsistente:
    raise ValueError('attack_cat no se corresponde 1:1 con Label; revisar limpieza.')
print('Verificación target multiclase OK: Benign <=> Label=0 al 100%.')

for col in ['state', 'service', 'attack_cat']:
    df[col] = df[col].astype('category')

## 2. Imputación de los blancos estructurales

`is_ftp_login`, `ct_ftp_cmd` y `ct_flw_http_mthd` solo tienen sentido cuando el flujo pertenece
a FTP o HTTP. Su ausencia es por diseño, **pero la redistribución de Kaggle codifica esa
ausencia de forma inconsistente**: el archivo 1 escribe `0` donde los archivos 2, 3 y 4
escriben un blanco.

Además el blanco no sigue al protocolo, que es lo que cabría esperar si fuera estructural:

| Condición | `is_ftp_login` en blanco |
|---|---|
| `service != ftp` | 57.69 % |
| `service == ftp` | 37.49 % |

Es decir, el blanco identifica el **archivo de procedencia**, no el tráfico. Codificarlo como
una categoría `No aplica` introduciría una característica que permite deducir de qué archivo
salió cada fila, es decir, fuga de procedencia en lugar de señal de tráfico.

**Decisión: imputar a 0.** El blanco y el cero expresan la misma semántica, `service` ya
codifica el protocolo en one-hot, y el valor `1` (que sí discrimina) está codificado de forma
homogénea en los cuatro archivos.

In [ ]:
BLANCOS_ESTRUCTURALES = ['is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd']

filas_blanco = {}
for col in BLANCOS_ESTRUCTURALES:
    nulos = int(df[col].isna().sum())
    filas_blanco[col] = nulos
    # is_ftp_login es 'boolean' nullable y requiere False, no 0.
    relleno = False if df[col].dtype == 'boolean' else 0
    df[col] = df[col].fillna(relleno)
    print(f'{col:18s} {nulos:>9,} blancos -> 0')

display(pd.DataFrame({
    'variable': list(filas_blanco),
    'blancos': list(filas_blanco.values()),
    'pct_del_total': [round(v / N * 100, 2) for v in filas_blanco.values()],
}))

## 3. Orden temporal y partición 75/15/10

Las variables `ct_*` cuentan eventos en una ventana de 100 conexiones **retrospectiva**,
terminada en el `Ltime` del propio flujo. En una partición aleatoria, los flujos de una misma
ráfaga quedarían repartidos entre train y test, y el modelo podría acertar no porque detecte
una intrusión sino porque memorizó la composición de la ráfaga.

Por eso se particiona por **orden temporal estricto**: todo lo que está en prueba ocurrió
después de todo lo que está en entrenamiento.

### El alineamiento de los cortes

Calcular el corte en el 75 % de las filas no basta. Las filas byte-idénticas comparten
`Ltime`, de modo que un corte que caiga entre dos registros con la misma marca de tiempo las
separaría, y eso es justamente la fuga que la partición temporal debe cerrar. Por eso cada
corte se desplaza hacia adelante hasta alcanzar un límite de marca de tiempo.

In [ ]:
df = df.sort_values('Ltime', kind='stable').reset_index(drop=True)
ltime = df['Ltime'].to_numpy()
n = len(df)


def corte_alineado(fraccion):
    # Índice de corte desplazado hasta un límite de Ltime.
    # Sin este desplazamiento, el corte puede partir un grupo de filas byte-idénticas
    # entre dos particiones.
    i = int(fraccion * n)
    while 0 < i < n and ltime[i] == ltime[i - 1]:
        i += 1
    return i


corte_val = corte_alineado(0.75)
corte_test = corte_alineado(0.90)

idx_train = np.arange(0, corte_val)
idx_val = np.arange(corte_val, corte_test)
idx_test = np.arange(corte_test, n)

print(f'corte train|val en fila {corte_val:,}  (Ltime {pd.to_datetime(ltime[corte_val], unit="s")})')
print(f'corte val|test  en fila {corte_test:,}  (Ltime {pd.to_datetime(ltime[corte_test], unit="s")})')
print(f'filas desplazadas por el alineamiento: {corte_val - int(0.75 * n)} y {corte_test - int(0.90 * n)}')
print(f'train={len(idx_train):,}  val={len(idx_val):,}  test={len(idx_test):,}')

In [ ]:
# Composición de cada partición. Las diez categorías están presentes en las tres, pero la
# proporción de Benign no es homogénea: la composición del tráfico cambia entre las dos
# sesiones de captura del dataset. No es un defecto del preprocesamiento sino una
# propiedad del dataset, y se analiza en la discusión de resultados.
part = pd.Series('train', index=df.index)
part.iloc[idx_val] = 'val'
part.iloc[idx_test] = 'test'

display(pd.DataFrame({
    'particion': ['train', 'val', 'test'],
    'n': [len(idx_train), len(idx_val), len(idx_test)],
    'n_clases': [df.iloc[i]['attack_cat'].nunique() for i in (idx_train, idx_val, idx_test)],
    'benign_pct': [round(100 * (df.iloc[i]['attack_cat'] == 'Benign').mean(), 2)
                   for i in (idx_train, idx_val, idx_test)],
}))

## 4. Imputación por moda, ajustada solo con entrenamiento

Quedan 304 nulos en `dsport`, 8 en `sport` y 8 en `state`, todos por tokens corruptos. Se
imputan por la moda.

La moda se estima **exclusivamente sobre el conjunto de entrenamiento**. Calcularla sobre las
2.54 M filas completas habría incorporado información de validación y prueba en la
transformación del conjunto de entrenamiento, lo que constituye una forma de fuga de
información.

Obsérvese que estas imputaciones se aplican *después* de la partición precisamente por eso.

In [ ]:
IMPUTAR_MODA = ['dsport', 'sport', 'state']
train_idx = df.index[idx_train]

modas = {}
for col in IMPUTAR_MODA:
    nulos = int(df[col].isna().sum())
    moda = df.loc[train_idx, col].dropna().mode().iat[0]
    modas[col] = moda.item() if hasattr(moda, 'item') else moda
    df[col] = df[col].fillna(moda)
    print(f'{col:8s} {nulos:>4,} nulos -> moda(train)={modas[col]!r}')

## 5. Ingeniería de características: rangos de puerto

`sport` y `dsport` tienen 64 598 y 64 627 valores únicos. Un one-hot daría 129 225 categorías,
inviable. Pero el problema relevante no es computacional sino estadístico:

| Clase | Puertos distintos |
|---|---|
| `Benign` | 64 581 |
| `Exploits` | 1 507 |
| `Generic` | 79 |
| `Fuzzers` | 12 |
| `Analysis` | 3 |

Los ataques sintéticos se emitieron siempre sobre los mismos puertos. Con el valor crudo el
modelo memoriza «puerto 6881 → Generic», asociación que no se sostiene en una red real. Es el
mismo atajo no generalizable que se descarta para las direcciones IP.

Se conservan por tanto dos señales de baja cardinalidad por puerto: si es efímero (≥ 1024) y
su franja (bien conocido, registrado, dinámico).

In [ ]:
def banda(p):
    if p <= 1023:
        return 'bien_conocido'
    if p <= 49151:
        return 'registrado'
    return 'dinamico'


for pre in ('sport', 'dsport'):
    col = df[pre]
    df[f'{pre}_efimero'] = (col >= 1024).astype('Int8')
    df[f'{pre}_franja'] = pd.Categorical(
        np.where(col.isna(), 'desconocido', col.map(banda)),
        categories=['bien_conocido', 'registrado', 'dinamico', 'desconocido'],
    )
    print(f'{pre}_efimero: {100 * float(df[f"{pre}_efimero"].astype("float64").mean()):.2f}% '
          f'de puertos efímeros')
    print(f'{pre}_franja:  {dict(df[f"{pre}_franja"].value_counts())}')

## 6. Selección de columnas

Se eliminan ocho columnas:

| Variable | Motivo |
|---|---|
| `attack_cat` | Es el objetivo |
| `Label` | Versión binaria del objetivo (Benign ⟺ Label=0) |
| `srcip`, `dstip` | Identidad de host: 43 y 47 valores únicos permiten asociar una IP a una familia de ataque |
| `Stime`, `Ltime` | Marcas de tiempo absolutas: describen el instante, no el comportamiento |
| `sport`, `dsport` | 129 225 categorías en conjunto; sustituidas por los derivados de franja |

In [ ]:
ELIMINAR = {
    'attack_cat': 'variable objetivo',
    'Label': 'versión binaria del objetivo (Benign <=> Label=0)',
    'srcip': 'identidad de host, 43 valores únicos: memoriza hosts',
    'dstip': 'identidad de host, 47 valores únicos: memoriza hosts',
    'Stime': 'marca de tiempo absoluta: correlaciona con la sesión de captura',
    'Ltime': 'marca de tiempo absoluta: correlaciona con la sesión de captura',
    'sport': '129 225 categorías junto a dsport: se derivan rangos',
    'dsport': '129 225 categorías junto a sport: se derivan rangos',
}

X = df.drop(columns=list(ELIMINAR)).copy()
print(f'{len(ELIMINAR)} columnas eliminadas -> {X.shape[1]} conservadas '
      f'(incluye 4 derivadas de puerto)')
assert 'attack_cat' not in X.columns and 'Label' not in X.columns

# Registro de la acción aplicada a cada columna, para el repositorio.
acciones = []
for col in df.columns:
    if col in ELIMINAR:
        acciones.append({'columna': col, 'accion': 'eliminar', 'motivo': ELIMINAR[col]})
    elif col.endswith(('_efimero', '_franja')):
        acciones.append({'columna': col, 'accion': 'derivar',
                         'motivo': 'ingeniería de puerto (baja cardinalidad)'})
    elif col == 'attack_cat':
        continue
    elif col in ('proto', 'state', 'service'):
        acciones.append({'columna': col, 'accion': 'one-hot',
                         'motivo': 'categórica de baja cardinalidad'})
    elif col in BLANCOS_ESTRUCTURALES or col in IMPUTAR_MODA:
        acciones.append({'columna': col, 'accion': 'imputar',
                         'motivo': 'blanco estructural o token corrupto'})
    elif col == 'is_sm_ips_ports':
        acciones.append({'columna': col, 'accion': 'binaria',
                         'motivo': 'indicador predefinido'})
    else:
        acciones.append({'columna': col, 'accion': 'numerica',
                         'motivo': 'log1p condicional + estandarización'})

tabla_columnas = pd.DataFrame(acciones)
tabla_columnas.to_csv(OUT_DIR / 'preprocesamiento_columnas.csv', index=False)
display(tabla_columnas.groupby('accion').size().rename('columnas').to_frame())

## 7. Transformación logarítmica

En `sbytes`, por ejemplo, la media (4 340) supera en tres órdenes de magnitud a la mediana
(1 470). Una estandarización ordinaria no corrige esa condición: al dividir por la desviación
estándar, dominada por unos pocos valores extremos, comprime a la mayoría de las
observaciones en un intervalo mínimo, y el aprendizaje pasa a depender de la magnitud del flujo
en lugar de su comportamiento.

Se aplica `log(1+x)` a toda variable con asimetría $|s| > 1$. Se usa `log(1+x)` y no `log(x)`
porque los ceros son plentiful en este dataset: hasta el 96.07 % en `ct_ftp_cmd`.

Dos precisiones:

1. **La asimetría se mide solo sobre train.** Calcularla sobre las 2.54 M filas habría
   incorporado estadísticas de test en la transformación de train.
2. **La transformación solo importa a dos de los tres modelos.** Es necesaria para la regresión
   logística y el SVM, que optimizan sobre el espacio de variables, pero es indiferente para un
   Random Forest, que decide mediante umbrales y es invariante a cualquier transformación
   monótona.

In [ ]:
UMBRAL_SKEW = 1.0
num_cols = X.select_dtypes(include=['number']).columns.tolist()

skew = pd.DataFrame({
    'variable': num_cols,
    'skew_train': [round(float(X.loc[train_idx, c].astype('float64').skew()), 4) for c in num_cols],
})
skew['transformar'] = skew['skew_train'].abs() > UMBRAL_SKEW
skew['skew_tras_log1p'] = [
    round(float(np.log1p(X.loc[train_idx, c].astype('float64').clip(lower=0))
                 .replace([np.inf, -np.inf], np.nan).skew()), 4)
    for c in skew['variable']
]
skew = skew.sort_values('skew_train', ascending=False).reset_index(drop=True)

LOG1P_COLS = skew.loc[skew['transformar'], 'variable'].tolist()
print(f'asimetría medida sobre {len(train_idx):,} filas de train')
print(f'umbral |skew| > {UMBRAL_SKEW} -> se transforman {len(LOG1P_COLS)} de {len(skew)} numéricas')
print(f'no se transforman: {skew.loc[~skew["transformar"], "variable"].tolist()}')

skew.to_csv(OUT_DIR / 'preprocesamiento_log1p.csv', index=False)
display(skew.assign(ganancia=(skew.skew_tras_log1p.abs() - skew.skew_train.abs()))
       .sort_values('ganancia').head(8)
       [['variable', 'skew_train', 'skew_tras_log1p', 'ganancia']])

In [ ]:
for col in LOG1P_COLS:
    X[col] = np.log1p(X[col].astype('float64').clip(lower=0)).astype('float32')
print(f'log1p aplicado a {len(LOG1P_COLS)} columnas.')

## 8. Persistencia en parquet

El formato parquet es obligatorio y no una preferencia. Los `dtypes` de ancho reducido
(`Int8`, `Int16`, `Float32`, `boolean`, `category`) son los que permiten que el conjunto quepa
en memoria; un CSV los destruiría al releerlo, y además haría reaparecer los tokens corruptos
`2` y `4` de `is_ftp_login` como si fueran valores legítimos. Con parquet el esquema se
conserva exactamente y Spark lo lee de forma nativa para la etapa de modelado.

In [ ]:
PROC_DIR.mkdir(parents=True, exist_ok=True)
for split, idx in (('train', idx_train), ('val', idx_val), ('test', idx_test)):
    destino = PROC_DIR / f'{split}.parquet'
    X.iloc[idx].to_parquet(destino, index=False, compression='snappy')
    print(f'{split:5s} {len(idx):>9,} filas x {X.shape[1]} columnas '
          f'-> {destino.stat().st_size / 1e6:7.1f} MB')

In [ ]:
metadatos = {
    'filas_totales': int(n),
    'duplicados_conservados': int(df.duplicated().sum()),
    'fracciones': {'train': 0.75, 'val': 0.15, 'test': 0.10},
    'corte_train_val': int(corte_val),
    'corte_val_test': int(corte_test),
    'Ltime_corte_train_val': str(pd.to_datetime(ltime[corte_val], unit='s')),
    'Ltime_corte_val_test': str(pd.to_datetime(ltime[corte_test], unit='s')),
    'modas_imputadas': modas,
    'columnas_eliminadas': ELIMINAR,
    'umbral_skew': UMBRAL_SKEW,
    'variables_log1p': LOG1P_COLS,
    'blancos_estructurales_imputados_a_cero': BLANCOS_ESTRUCTURALES,
    'n_columnas_modelo': int(X.shape[1]),
}
with open(PROC_DIR / 'metadatos.json', 'w') as fh:
    json.dump(metadatos, fh, indent=2, ensure_ascii=False)
print('metadatos guardados en', PROC_DIR / 'metadatos.json')

## 9. Verificaciones

Seis comprobaciones que deben satisfacerse para dar por válida la salida. Están planteadas
como aserciones: si alguna falla, la celda se detiene y el error queda explícito.

In [ ]:
print('[V1] las diez categorías presentes en cada partición')
n_clases = {p: df.iloc[i]['attack_cat'].nunique()
            for p, i in (('train', idx_train), ('val', idx_val), ('test', idx_test))}
print('   ', n_clases)
assert all(v == 10 for v in n_clases.values()), 'Falta alguna clase en una partición'

In [ ]:
print('[V2] clases minoritarias por partición')
menores = df['attack_cat'].value_counts().tail(4).index.tolist()
conteo = pd.DataFrame({
    p: df.iloc[i]['attack_cat'].value_counts().reindex(menores).fillna(0).astype(int)
    for p, i in (('train', idx_train), ('val', idx_val), ('test', idx_test))
})
conteo.index.name = 'clase'
conteo.to_csv(OUT_DIR / 'preprocesamiento_clases_minoritarias.csv')
display(conteo)
print('   Worms queda con 24 filas en prueba: se reportará con la advertencia de')
print('   incertidumbre correspondiente, sin usarla para comparar modelos.')

In [ ]:
print('[V3] solapamiento de la ventana ct_* (100 conexiones previas)')
es_test = np.zeros(n, dtype=bool); es_test[idx_test] = True
es_train = np.zeros(n, dtype=bool); es_train[idx_train] = True
acum = np.concatenate([[0], np.cumsum(es_train)])
i = np.where(es_test)[0]
lo = np.maximum(i - 100, 0)
ancho = np.maximum(np.minimum(i, 100), 1)
pct = 100 * float(np.mean((acum[i] - acum[lo]) / ancho))
print(f'   {pct:.2f}% del contexto previo de una fila de prueba está en train')
assert pct == 0.0, 'Hay solapamiento temporal: la partición no es cronológica'

In [ ]:
print('[V4] ningún corte cae dentro de un empate de Ltime')
for nombre, corte in (('train|val', corte_val), ('val|test', corte_test)):
    distinto = bool(ltime[corte] != ltime[corte - 1])
    print(f'   corte {nombre}: Ltime distinto a ambos lados = {distinto}')
    assert distinto, f'El corte {nombre} parte un grupo de Ltime idéntico'

In [ ]:
print('[V5] ninguna fila idéntica cruza de partición')
_orden = np.empty(n, dtype=np.int8)
_orden[idx_train], _orden[idx_val], _orden[idx_test] = 0, 1, 2
_hash = pd.util.hash_pandas_object(df.assign(_part=_orden).drop(columns='_part'), index=False)
_cruce = int((pd.DataFrame({'h': _hash, 'p': _orden}).groupby('h')['p'].nunique() > 1).sum())
print(f'   grupos de filas idénticas repartidos entre particiones: {_cruce}')
assert _cruce == 0, 'Hay filas idénticas en particiones distintas'

In [ ]:
print('[V6] sin fuga de la etiqueta')
assert 'attack_cat' not in X.columns and 'Label' not in X.columns
print('    attack_cat y Label ausentes del espacio de características: OK')

print()
print('Preprocesamiento completado.')
print(f'  filas conservadas : {n:,} (duplicados NO eliminados: {int(df.duplicated().sum()):,})')
print(f'  columnas          : 49 -> {X.shape[1]}')
print(f'  particiones       : {PROC_DIR}/{{train,val,test}}.parquet')